# 5-3. 前処理

In [ ]:
import numpy as np
import scipy as sp
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns
sns.set()
%matplotlib inline

%precision 3

## 5-3-1. 使用するデータ・データの関係性

In [ ]:
cd sampledata

In [ ]:
customer_master = pd.read_csv('customer_master.csv')
customer_master.head()

In [ ]:
transaction_1 = pd.read_csv('transaction_1.csv')
transaction_1.head()

In [ ]:
transaction_detail_1 = pd.read_csv('transaction_detail_1.csv')
transaction_detail_1.head()

In [ ]:
# transaction_1の続きになっている
transaction_2 = pd.read_csv('transaction_2.csv')
transaction_2.head()

In [ ]:
# transaction_detail_1の続きになっている
transaction_detail_2 = pd.read_csv('transaction_detail_2.csv')
transaction_detail_2.head()

## 5-3-2. データの連結

In [ ]:
# データを連結する
# ignore_index = True → indexを振り直す
transaction = pd.concat([transaction_1, transaction_2], ignore_index=True)
transaction.head()

In [ ]:
print(len(transaction_1))
print(len(transaction_2))
print(len(transaction))

In [ ]:
# 同様に連結する
transaction_detail=pd.concat([transaction_detail_1,transaction_detail_2], ignore_index=True)
transaction_detail.head()

In [ ]:
print(len(transaction_detail_1))
print(len(transaction_detail_2))
print(len(transaction_detail))

## 5-3-3. データの結合

In [ ]:
# 取引詳細データと取引データを結合する
join_data = pd.merge(transaction_detail, transaction, on="transaction_id", how="left")
join_data.head()

In [ ]:
len(join_data)

In [ ]:
# 顧客データを結合する
join_data = pd.merge(join_data, customer_master, on="customer_id", how="left")
join_data.head()

## 5-3-4. データの整備

In [ ]:
# 商品名が統一されていない
print(pd.unique(join_data["item_name"]))
print(len(pd.unique(join_data["item_name"])))

In [ ]:
# 商品名のフォーマットを統一する
join_data["item_name"] = join_data["item_name"].str.upper()
join_data["item_name"] = join_data["item_name"].str.replace("　", "")
join_data["item_name"] = join_data["item_name"].str.replace(" ", "")

In [ ]:
join_data.head()

In [ ]:
print(pd.unique(join_data["item_name"]))
print(len(pd.unique(join_data["item_name"])))

In [ ]:
# ソートすることもできる
#transaction_detail_1.sort_values(by=["item_name"], ascending=True)

## 5-3-5. 欠損値の確認・処理

In [ ]:
#欠損値の確認
join_data.isnull().any(axis=0)

In [ ]:
# 年齢の欠損を埋める(ここでは平均を使う)
join_data['age'] = join_data['age'].fillna(int(join_data['age'].mean()))

In [ ]:
# 年齢の欠損がなくなっていることを確認する
join_data.head()

In [ ]:
# priceの欠損値を埋める
flg_is_null = join_data["item_price"].isnull()
for trg in list(join_data.loc[flg_is_null, "item_name"].unique()):
    price = join_data.loc[(~flg_is_null) & (join_data["item_name"] == trg), "item_price"].max()
    join_data.loc[(flg_is_null) & (join_data["item_name"] == trg), ["item_price"]] = price
join_data

In [ ]:
# Nanがなくなっていることを確認する
join_data.isnull().any(axis=0)

In [ ]:
# 金額が埋められているかを確認する
for trg in list(join_data["item_name"].sort_values().unique()):
    print(trg + "の最大額：" + str(join_data.loc[join_data["item_name"]==trg]["item_price"].max()) + "の最小額：" + str(join_data.loc[join_data["item_name"]==trg]["item_price"].min(skipna=False)))

## 5-3-6. データの削除

In [ ]:
# データの削除
join_data.drop('customer_id', axis=1, inplace=True)

In [ ]:
join_data